# PCS + Legat AI Avatar — Kaggle GPU smoke test

Этот notebook сначала проверяет бесплатный GPU, затем делает короткий тест: TTS → EchoMimicV2 → MuseTalk 1.5 → FFmpeg → ffprobe.

**Не публикуй портрет и реальные материалы в output notebook.** Notebook должен оставаться private.


In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys, time

WORK = Path('/kaggle/working')
OUTPUT = WORK / 'output'
OUTPUT.mkdir(exist_ok=True)
PRIVATE_ROOTS = [
    Path('/kaggle/input/pcs-legat-private-assets'),
    Path('/kaggle/input/pcs-legat-assets'),
]
SMOKE_SECONDS = 5
print('Python:', sys.version)
print('Output:', OUTPUT)


In [ ]:
# GPU gate: do not continue without enough VRAM.
if shutil.which('nvidia-smi') is None:
    raise RuntimeError('GPU не включён. Notebook settings → Accelerator → GPU.')
subprocess.run(['nvidia-smi'], check=True)
mem = float(subprocess.check_output([
    'nvidia-smi','--query-gpu=memory.total','--format=csv,noheader,nounits'
], text=True).splitlines()[0])
if mem < 14000:
    raise RuntimeError(f'Недостаточно VRAM: {mem/1024:.1f} GB. Нужен GPU около 16 GB или больше.')


In [ ]:
# Find private avatar without hard-coding a public path.
def find_asset(name: str) -> Path:
    for root in PRIVATE_ROOTS:
        if root.exists():
            matches = list(root.rglob(name))
            if matches:
                return matches[0]
    raise FileNotFoundError(
        f'{name} не найден. Создай private Kaggle Dataset pcs-legat-private-assets и добавь файл.'
    )

AVATAR = find_asset('6738.png')
print('Avatar found:', AVATAR)


In [ ]:
# Base packages and free test voices.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'pyyaml>=6.0.2', 'edge-tts>=7.0.0', 'huggingface_hub>=0.30'], check=True)

RU_WAV = WORK / 'voice_ru.wav'
EN_WAV = WORK / 'voice_en.wav'
ru_text = 'Перед арендой машины снимите кузов и салон одним непрерывным видео.'
en_text = 'Before renting a car, record the body and interior in one continuous video.'

# Edge TTS is used only for the free smoke test. Later replace with approved stable voices.
subprocess.run(['edge-tts', '--voice', 'ru-RU-DmitryNeural', '--text', ru_text, '--write-media', str(WORK/'voice_ru.mp3')], check=True)
subprocess.run(['edge-tts', '--voice', 'en-US-GuyNeural', '--text', en_text, '--write-media', str(WORK/'voice_en.mp3')], check=True)
for src, dst in [(WORK/'voice_ru.mp3', RU_WAV), (WORK/'voice_en.mp3', EN_WAV)]:
    subprocess.run(['ffmpeg','-y','-i',str(src),'-t',str(SMOKE_SECONDS),'-ar','16000','-ac','1','-c:a','pcm_s16le',str(dst)], check=True)


In [ ]:
# Install EchoMimicV2 from its official repository.
ECHO = WORK / 'echomimic_v2'
if not ECHO.exists():
    subprocess.run(['git','clone','--depth','1','https://github.com/antgroup/echomimic_v2',str(ECHO)], check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(ECHO/'requirements.txt')], check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','--no-deps','facenet_pytorch==2.6.0'], check=True)

WEIGHTS = ECHO / 'pretrained_weights'
if not WEIGHTS.exists():
    subprocess.run(['git','clone','--depth','1','https://huggingface.co/BadToBest/EchoMimicV2',str(WEIGHTS)], check=True)
print('EchoMimicV2 ready')


In [ ]:
# Patch the official accelerated config with the private avatar and short RU audio.
import yaml
source_cfg = ECHO / 'configs/prompts/infer_acc.yaml'
cfg = yaml.safe_load(source_cfg.read_text())
cfg['test_cases'] = {str(AVATAR): [str(RU_WAV)]}
# Keep the test short where these keys exist in the upstream config.
for key in ('L','length','video_length','num_frames'):
    if key in cfg:
        cfg[key] = min(int(cfg[key]), 120)
custom_cfg = WORK / 'echo_smoke.yaml'
custom_cfg.write_text(yaml.safe_dump(cfg, sort_keys=False), encoding='utf-8')
print(custom_cfg.read_text()[:4000])


In [ ]:
# Semi-body animation. On a free P100/T4 this can take a long time.
echo_started = time.time()
subprocess.run([sys.executable, 'infer_acc.py', '--config', str(custom_cfg)], cwd=ECHO, check=True)
print('Echo seconds:', round(time.time()-echo_started, 1))

mp4s = list(ECHO.rglob('*.mp4'))
if not mp4s:
    raise RuntimeError('EchoMimicV2 completed but produced no MP4.')
BASE = max(mp4s, key=lambda p:p.stat().st_mtime)
BASE_COPY = OUTPUT/'avatar_base.mp4'
shutil.copy2(BASE, BASE_COPY)
print('Base:', BASE_COPY)


In [ ]:
# Install MuseTalk 1.5 and official weights.
MUSE = WORK / 'MuseTalk'
if not MUSE.exists():
    subprocess.run(['git','clone','--depth','1','https://github.com/TMElyralab/MuseTalk',str(MUSE)], check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(MUSE/'requirements.txt')], check=True)
download = MUSE/'download_weights.sh'
if download.exists():
    subprocess.run(['bash',str(download)], cwd=MUSE, check=True)
else:
    raise RuntimeError('MuseTalk download_weights.sh is missing; upstream layout changed.')


In [ ]:
# Run MuseTalk separately for RU and EN while keeping the same base visual.
def run_musetalk(audio: Path, language: str) -> Path:
    conf = MUSE / f'configs/inference/pcs_{language}.yaml'
    conf.write_text(
        'task_0:\n'
        f'  video_path: "{BASE_COPY}"\n'
        f'  audio_path: "{audio}"\n'
        '  bbox_shift: 0\n', encoding='utf-8'
    )
    before = {p: p.stat().st_mtime for p in MUSE.rglob('*.mp4')}
    subprocess.run(['bash','inference.sh','v1.5','normal'], cwd=MUSE, env={**os.environ, 'INFERENCE_CONFIG': str(conf)}, check=False)
    # The official shell script normally uses configs/inference/test.yaml; run module directly for deterministic config.
    subprocess.run([
        sys.executable,'-m','scripts.inference',
        '--inference_config',str(conf),
        '--result_dir',str(MUSE/'results/pcs'),
        '--unet_model_path',str(MUSE/'models/musetalkV15/unet.pth'),
        '--unet_config',str(MUSE/'models/musetalkV15/musetalk.json'),
        '--version','v15'
    ], cwd=MUSE, check=True)
    candidates = list((MUSE/'results/pcs').rglob('*.mp4'))
    if not candidates:
        raise RuntimeError(f'MuseTalk {language} produced no MP4.')
    return max(candidates, key=lambda p:p.stat().st_mtime)

RU_RAW = run_musetalk(RU_WAV, 'ru')
EN_RAW = run_musetalk(EN_WAV, 'en')
print(RU_RAW, EN_RAW)


In [ ]:
# Normalize delivery files to 1920×1080, 30 fps, H.264/AAC.
def normalize(src: Path, dst: Path):
    vf = 'scale=1920:1080:force_original_aspect_ratio=decrease,pad=1920:1080:(ow-iw)/2:(oh-ih)/2:black,fps=30,format=yuv420p'
    subprocess.run([
        'ffmpeg','-y','-i',str(src),'-t',str(SMOKE_SECONDS),'-vf',vf,
        '-c:v','libx264','-preset','medium','-crf','18','-c:a','aac','-b:a','192k',str(dst)
    ], check=True)

RU_FINAL = OUTPUT/'avatar_ru.mp4'
EN_FINAL = OUTPUT/'avatar_en.mp4'
normalize(RU_RAW, RU_FINAL)
normalize(EN_RAW, EN_FINAL)


In [ ]:
# Technical QA.
def probe(path: Path):
    raw = subprocess.check_output([
        'ffprobe','-v','error','-show_streams','-show_format','-of','json',str(path)
    ], text=True)
    return json.loads(raw)

report = {p.name: probe(p) for p in [BASE_COPY, RU_FINAL, EN_FINAL]}
(OUTPUT/'qa_report.json').write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')
manifest = {
    'status': 'smoke_test_completed',
    'avatar': AVATAR.name,
    'duration_target_seconds': SMOKE_SECONDS,
    'outputs': [p.name for p in [BASE_COPY, RU_FINAL, EN_FINAL]],
}
(OUTPUT/'run_manifest.json').write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')
print(json.dumps(manifest, ensure_ascii=False, indent=2))


## После smoke test

Не включай ежедневное расписание, пока вручную не проверены: лицо, губы, зубы, глаза, руки, дрейф личности и фактическая длительность. После успешной проверки `SMOKE_SECONDS` заменяется на 30, а тестовые тексты — на ежедневный `job.json`.
